# Simulating games of Hangman using a heuristic policy

In this notebook, we shall run example games of Hangman using our pre-trained transformer model as shown in the notebook `pre_train_notebook.ipynb` by constructing an intuitive heuristic policy.

In [1]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.35'

import torch
from utils.utils import TextDataset
from torch.utils.data import random_split

# Prepare the dataset
dataset = TextDataset('./data/words_250000_train.txt')

train_split_percent = 0.99
train_size = int(train_split_percent * len(dataset))
test_size = len(dataset) - train_size

print(f'Training size   : {train_size:,}')
print(f'Validation size : {test_size:,}')

# Using the same seed as training
train_dataset, val_dataset = random_split(
    dataset, [train_size, test_size], generator=torch.Generator().manual_seed(0)
)

val_words = [val_dataset[i] for i in range(len(val_dataset))]

Training size   : 225,027
Validation size : 2,273


### Importing custom Hangman gym-based env
Check out the source code of the environment under `env` directory. 
* Follows the gym protocol.
* Is vectorized and can support multithreading for parallel computation.

In [2]:
from env.hangman_jax import JaxHangmanEnv

In [3]:
# Initialize the vectorized JAX environment with the validation words
eval_env = JaxHangmanEnv(val_words)
num_val_games = eval_env.num_words
print(f'Loaded {num_val_games:,} validation games into GPU memory!')

Loaded 2,273 validation games into GPU memory!


### Implementing our Agent which will interact with the Hangman environment

In [8]:
import jax
import jax.numpy as jnp
from flax import nnx
from model_jax.Models import Transformer

class Agent(nnx.Module):
    def __init__(self, rngs):
        # Pretrained transformer model in Flax NNX
        self.pretrainedLLM = Transformer(
            src_vocab=28, d_model=128, max_seq_len=32, N=12, heads=8, dropout=0.0, rngs=rngs
        )
        self.pretrainedLLM.load_weights("weights/aaa_best_weights_nnx.npz")
        
    def act(self, x, action_mask):
        # 1. Forward pass
        logits = self.pretrainedLLM(x)                   # (B, 32, 28)
        probs = jax.nn.softmax(logits, axis=-1)           # (B, 32, 28)

        # 2. Aggregate letter probabilities row-wise across non-pad characters
        pad_mask = (x != 0)[:, :, None]                  # (B, 32, 1)
        probs_sum = jnp.sum(probs * pad_mask, axis=1)    # (B, 28)
        letter_probs = probs_sum[:, 1:27]                # (B, 26) for 'a'..'z'

        # 3. Mask out invalid (already-guessed) actions
        masked_probs = jnp.where(action_mask, letter_probs, 0.0)
        action = jnp.argmax(masked_probs, axis=-1)       # (B,) greedy heuristic
        return action


In [9]:
# Initialize the agent in evaluation mode
rngs = nnx.Rngs(0)
agent = Agent(rngs)
agent.eval()
print("Agent initialized with pre-trained weights.")

Agent initialized with pre-trained weights.


In [10]:
import time

# 1. Define the single-step rollout transition for scan
def rollout_step(states, _):
    obs = eval_env.observe(states)
    mask = eval_env.action_mask(states)
    actions = agent.act(obs, mask)
    next_states, rewards = eval_env.step(states, actions)
    return next_states, None

# 2. Compile full 26-step rollout across all games into a single XLA kernel
@jax.jit
def run_rollout(states):
    final_states, _ = jax.lax.scan(rollout_step, states, None, length=26)
    return final_states

# 3. Reset all validation games at once on GPU
all_indices = jnp.arange(num_val_games)
init_states = eval_env.reset_indices(all_indices)

# 4. JIT Warmup compilation
print("Compiling JAX rollout kernel on GPU...")
t_comp_start = time.perf_counter()
warmup_states = run_rollout(eval_env.reset_indices(all_indices[:32]))
warmup_states.won.block_until_ready()
t_comp = time.perf_counter() - t_comp_start
print(f"JIT Compilation finished in {t_comp:.2f} s")

# 5. Execute all validation games simultaneously on GPU!
start_time = time.perf_counter()
final_states = run_rollout(init_states)
final_states.won.block_until_ready()
end_time = time.perf_counter()

total_games = num_val_games
total_wins = int(jnp.sum(final_states.won))
win_rate = 100.0 * total_wins / total_games
elapsed = end_time - start_time

print('\n--------------------------------------------------------')
print(f'Total Games Evaluated \t: \t {total_games:,}')
print(f'Total Wins            \t: \t {total_wins:,}')
print(f'Win Rate              \t: \t {win_rate:.3f}%')
print(f'Execution Time        \t: \t {elapsed:.3f} s')
print(f'Throughput            \t: \t {total_games / elapsed:.1f} games/s')
print(f'Mean time per game    \t: \t {1000 * elapsed / total_games:.3f} ms')
print('--------------------------------------------------------')


Compiling JAX rollout kernel on GPU...
JIT Compilation finished in 2.22 s

--------------------------------------------------------
Total Games Evaluated 	: 	 2,273
Total Wins            	: 	 1,509
Win Rate              	: 	 66.388%
Execution Time        	: 	 4.828 s
Throughput            	: 	 470.8 games/s
Mean time per game    	: 	 2.124 ms
--------------------------------------------------------


### Conclusion
We can see that our model with the heuristic policy performs fairly well even on the validation dataset. However, one can further try improving the model by using Reinforcement Learning techniques. This can be a future area of research.